# Practical Demo for `ellphi-alpha`

This notebook is the canonical user-facing walkthrough with numeric checks and visualizations.


## Setup

In [1]:
from pathlib import Path
import sys

import numpy as np

# Make local package import reliable when running from repo root.
repo_root = Path.cwd()
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import ellphi
import ellphi_alpha as ea

np.set_printoptions(precision=6, suppress=True)

## Example A: Pairwise alpha correspondence

For a 2-vertex simplex, `alpha` from `ellphi_alpha` should match `ellphi.tangency(...).t**2`.

In [2]:
p = ellphi.coef_from_cov([0.0, 0.0], [[0.45, 0.05], [0.05, 0.25]])[0]
q = ellphi.coef_from_cov([1.2, 0.6], [[0.30, -0.02], [-0.02, 0.55]])[0]

pair = ea.solve_minimax_from_coefs(np.stack([p, q]), tol=1e-12)
tang = ellphi.tangency(p, q)

print("alpha({0,1}) from minimax =", pair.alpha)
print("t from ellphi =", tang.t)
print("t**2 from ellphi =", tang.t**2)
print("absolute difference =", abs(pair.alpha - tang.t**2))
print("circumcenter =", pair.circumcenter)
print("weights =", pair.weights)

alpha({0,1}) from minimax = 1.1588753467747448
t from ellphi = 1.076510727663568
t**2 from ellphi = 1.1588753467747446
absolute difference = 2.220446049250313e-16
circumcenter = [0.692044 0.228959]
weights = [0.542606 0.457394]


## Example B: Small anisotropic simplex (`|sigma| = 3`)

Compute minimax on a triangle with different SPD matrices and inspect the active set.

In [3]:
centers_B = np.array([
    [0.0, 0.0],
    [2.0, 0.25],
    [0.7, 1.75],
])
covs_B = np.array([
    [[0.55, 0.08], [0.08, 0.30]],
    [[0.40, -0.06], [-0.06, 0.70]],
    [[0.75, 0.12], [0.12, 0.45]],
])
mats_B = np.linalg.inv(covs_B)

res_B = ea.solve_minimax(mats_B, centers_B, tol=1e-11)
diff_B = res_B.circumcenter[np.newaxis, :] - centers_B
fvals_B = np.einsum("ki,kij,kj->k", diff_B, mats_B, diff_B)

print("alpha(triangle) =", res_B.alpha)
print("converged =", res_B.converged, "iterations =", res_B.n_iter)
print("circumcenter =", res_B.circumcenter)
print("weights =", res_B.weights)
print("active_set =", res_B.active_set)
print("f_i(circumcenter) =", fvals_B)
print("max(f_i) - alpha =", np.max(fvals_B) - res_B.alpha)

alpha(triangle) = 2.8148325408804484
converged = True iterations = 32
circumcenter = [0.964445 0.709509]
weights = [0.357457 0.307888 0.334655]
active_set = [0, 1, 2]
f_i(circumcenter) = [2.814833 2.814833 2.814833]
max(f_i) - alpha = 0.0


## Example C: Incremental filtration and top entries

Build a small filtration up to dimension 2 and inspect the first entries in filtration order.

In [4]:
centers_C = np.array([
    [0.0, 0.0],
    [1.8, 0.1],
    [0.3, 1.6],
    [1.4, 1.4],
    [0.9, 0.7],
])
covs_C = np.array([
    [[0.55, 0.05], [0.05, 0.30]],
    [[0.45, -0.03], [-0.03, 0.60]],
    [[0.70, 0.08], [0.08, 0.40]],
    [[0.35, 0.02], [0.02, 0.50]],
    [[0.50, 0.00], [0.00, 0.45]],
])
mats_C = np.linalg.inv(covs_C)

filtration = ea.build_incremental_filtration(mats_C, centers_C, max_dim=2)

print("total filtration entries =", len(filtration))
counts = {}
for entry in filtration:
    dim = len(entry.simplex) - 1
    counts[dim] = counts.get(dim, 0) + 1
print("counts by simplex dimension =", counts)

print("\nTop 12 entries (simplex, alpha):")
for idx, entry in enumerate(filtration[:12], start=1):
    print(f"{idx:2d}. {entry.simplex} -> alpha={entry.alpha:.8f}")

total filtration entries = 13
counts by simplex dimension = {0: 5, 1: 6, 2: 2}

Top 12 entries (simplex, alpha):
 1. (0,) -> alpha=0.00000000
 2. (1,) -> alpha=0.00000000
 3. (2,) -> alpha=0.00000000
 4. (3,) -> alpha=0.00000000
 5. (4,) -> alpha=0.00000000
 6. (3, 4) -> alpha=0.39648718
 7. (1, 4) -> alpha=0.58213779
 8. (2, 3) -> alpha=0.64150565
 9. (2, 4) -> alpha=0.67394835
10. (0, 4) -> alpha=0.67453365
11. (2, 3, 4) -> alpha=0.80589926
12. (1, 3) -> alpha=0.86589459


## Example D (optional): GUDHI adapter

If `gudhi` is available, export to a simplex tree and print a compact persistence summary.

In [5]:
from ellphi_alpha.backends import GudhiBackend

backend = GudhiBackend()
if backend.is_available():
    st = ea.to_gudhi_simplex_tree(filtration)
    h0 = backend.persistence_intervals(st, dimension=0)
    h1 = backend.persistence_intervals(st, dimension=1)

    print("GUDHI available: yes")
    print("H0 interval count =", len(h0))
    print("H1 interval count =", len(h1))

    finite_h1 = h1[np.isfinite(h1[:, 1])] if len(h1) else np.empty((0, 2))
    if len(finite_h1):
        lifetimes = finite_h1[:, 1] - finite_h1[:, 0]
        print("largest finite H1 lifetimes =", np.sort(lifetimes)[-3:][::-1])
    else:
        print("No finite H1 intervals in this tiny example.")
else:
    print("GUDHI available: no")
    print("Skipping conversion. Install optional demo deps: poetry install --with demo")

GUDHI available: yes
H0 interval count = 5
H1 interval count = 2
largest finite H1 lifetimes = [0.131951 0.000094]


/Users/uda/Library/Caches/pypoetry/virtualenvs/ellphi-alpha-EUCoFB4J-py3.12/lib/python3.12/site-packages/gudhi/persistence_graphical_tools.py:277: SyntaxWarning: invalid escape sequence '\l'
  """This function plots the persistence diagram from persistence values list, a np.array of shape (N x 2)
